In [1]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "fitbit_prostate_cancer" for domain "fitbit_activity" and was generated for All of Us Controlled Tier Dataset v8
dataset_31237560_fitbit_activity_sql <- paste("
    SELECT
        activity_summary.person_id,
        activity_summary.date,
        activity_summary.activity_calories,
        activity_summary.calories_bmr,
        activity_summary.calories_out,
        activity_summary.elevation,
        activity_summary.fairly_active_minutes,
        activity_summary.floors,
        activity_summary.lightly_active_minutes,
        activity_summary.marginal_calories,
        activity_summary.sedentary_minutes,
        activity_summary.steps,
        activity_summary.very_active_minutes 
    FROM
        `activity_summary` activity_summary   
    WHERE
        activity_summary.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_activity_summary = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_plus_device_information = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_heart_rate_summary = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_heart_rate_level = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_steps_intraday = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_sleep_daily_summary = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_sleep_level = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_device_information = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `cb_criteria` cr       
                        WHERE
                            concept_id IN (4163261, 200970)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria ) 
            AND cb_search_person.person_id NOT IN (SELECT
                person_id 
            FROM
                `person` p 
            WHERE
                sex_at_birth_concept_id IN (45878463) ) )", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
fitbit_activity_31237560_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "fitbit_activity_31237560",
  "fitbit_activity_31237560_*.csv")
message(str_glue('The data will be written to {fitbit_activity_31237560_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_31237560_fitbit_activity_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  fitbit_activity_31237560_path,
  destination_format = "CSV")



── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260420/fitbit_activity_31237560/fitbit_activity_31237560_*.csv. Use this path when reading the data into your notebooks in the future.



In [2]:
# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {fitbit_activity_31237560_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
pca_activity <- read_bq_export_from_workspace_bucket(fitbit_activity_31237560_path)

dim(pca_activity)

head(pca_activity, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260420/fitbit_activity_31237560/fitbit_activity_31237560_000000000000.csv.



[1] 449924     13

person_id,date,activity_calories,calories_bmr,calories_out,elevation,fairly_active_minutes,floors,lightly_active_minutes,marginal_calories,sedentary_minutes,steps,very_active_minutes
<dbl>,<date>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2028613,2023-07-17,0,1402,2009,NA,0,NA,0,0,1440,4573,0
2028613,2023-08-18,0,1402,2107,NA,0,NA,0,0,1440,6443,0
2028613,2023-08-01,0,1402,2129,NA,0,NA,0,0,1440,6379,0
2028613,2023-08-04,0,1402,1853,NA,0,NA,0,0,1440,4120,0
2028613,2023-06-28,421,1402,1801,NA,0,NA,148,178,895,2436,0


In [4]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "fitbit_prostate_cancer" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_31237560_person_sql <- paste("
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth,
        person.self_reported_category_concept_id,
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `person` person 
    LEFT JOIN
        `concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_activity_summary = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_plus_device_information = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_heart_rate_summary = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_heart_rate_level = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_steps_intraday = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_sleep_daily_summary = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_sleep_level = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_device_information = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `cb_criteria` cr       
                        WHERE
                            concept_id IN (4163261, 200970)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria ) 
            AND cb_search_person.person_id NOT IN (SELECT
                person_id 
            FROM
                `person` p 
            WHERE
                sex_at_birth_concept_id IN (45878463) ) )", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
person_31237560_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "person_31237560",
  "person_31237560_*.csv")
message(str_glue('The data will be written to {person_31237560_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_31237560_person_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  person_31237560_path,
  destination_format = "CSV")

The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260427/person_31237560/person_31237560_*.csv. Use this path when reading the data into your notebooks in the future.



In [5]:
# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {person_31237560_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(gender = col_character(), race = col_character(), ethnicity = col_character(), sex_at_birth = col_character(), self_reported_category = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
pca_meta <- read_bq_export_from_workspace_bucket(person_31237560_path)

dim(pca_meta)

head(pca_meta, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260427/person_31237560/person_31237560_000000000000.csv.



[1] 627  12

person_id,gender_concept_id,gender,date_of_birth,race_concept_id,race,ethnicity_concept_id,ethnicity,sex_at_birth_concept_id,sex_at_birth,self_reported_category_concept_id,self_reported_category
<dbl>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>
2967502,45880669,Male,1953-06-15 00:00:00 UTC,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,45880669,Male,45882607,None of these
9633032,45880669,Male,1938-06-15 00:00:00 UTC,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,45880669,Male,45882607,None of these
5798599,45880669,Male,1954-06-15 00:00:00 UTC,903096,PMI: Skip,903096,PMI: Skip,45880669,Male,903096,PMI: Skip
1561089,45880669,Male,1948-06-15 00:00:00 UTC,903096,PMI: Skip,903096,PMI: Skip,45880669,Male,903096,PMI: Skip
6647893,45880669,Male,1956-06-15 00:00:00 UTC,903096,PMI: Skip,903096,PMI: Skip,45880669,Male,903096,PMI: Skip


In [10]:
source('/home/jupyter/workspaces/temporaldigitalhealthphenotyping/ind_descriptive_func.R')


Attaching package: ‘scales’


The following object is masked from ‘package:purrr’:

    discard


The following object is masked from ‘package:readr’:

    col_factor


corrplot 0.95 loaded



In [3]:
wear_compliance_func(path.out = '/home/jupyter/workspaces/result/Prostate/', df = pca_activity)


Attaching package: ‘scales’


The following object is masked from ‘package:purrr’:

    discard


The following object is masked from ‘package:readr’:

    col_factor


corrplot 0.95 loaded



── Wear compliance ──
     n_days       followup_days   
 Min.   :   4.0   Min.   :   4.0  
 1st Qu.:  94.0   1st Qu.:  98.0  
 Median : 149.0   Median : 212.0  
 Mean   : 717.6   Mean   : 909.5  
 3rd Qu.:1201.0   3rd Qu.:1564.5  
 Max.   :3967.0   Max.   :4443.0  


`geom_smooth()` using formula = 'y ~ x'


   person_id           n_days         first_date           last_date         
 Min.   :1002338   Min.   :   4.0   Min.   :2011-08-03   Min.   :2017-08-09  
 1st Qu.:1878468   1st Qu.:  94.0   1st Qu.:2018-12-26   1st Qu.:2023-09-22  
 Median :3188701   Median : 149.0   Median :2023-02-12   Median :2023-10-01  
 Mean   :4166346   Mean   : 717.6   Mean   :2021-01-19   Mean   :2023-07-16  
 3rd Qu.:6321677   3rd Qu.:1201.0   3rd Qu.:2023-06-22   3rd Qu.:2023-10-01  
 Max.   :9962965   Max.   :3967.0   Max.   :2023-09-23   Max.   :2023-10-01  
 followup_days     missing_days     pct_missing              compliance_tier
 Min.   :   4.0   Min.   :   0.0   Min.   : 0.000   High (>90%)      :440   
 1st Qu.:  98.0   1st Qu.:   0.0   1st Qu.: 0.000   Moderate (70-90%):107   
 Median : 212.0   Median :   5.0   Median : 1.961   Low (50-70%)     : 35   
 Mean   : 909.5   Mean   : 191.9   Mean   :11.550   Poor (<50%)      : 45   
 3rd Qu.:1564.5   3rd Qu.:  85.0   3rd Qu.:13.631                    

NULL

In [12]:
patient_stats <- activity_prep_stats(
    path.out = '/home/jupyter/workspaces/Data/',
    cohort = 'Prostate',
    df = pca_activity_sex,
    meta.data = pca_meta[,c('person_id', 'sex_at_birth')]
)


# A tibble: 6 × 9
  metric                     N   Mean     SD Median     Q1     Q3      P5    P95
  <chr>                  <int>  <dbl>  <dbl>  <dbl>  <dbl>  <dbl>   <dbl>  <dbl>
1 calories_out             627 2529.   438.  2507.  2.24e3 2797.  1.86e+3 3.26e3
2 fairly_active_minutes    627   18.6   15.1   15.6 7.49e0   26.5 1.50e+0 4.79e1
3 lightly_active_minutes   627  173.    67.0  173.  1.28e2  218.  6.59e+1 2.83e2
4 sedentary_minutes        627  882.   197.   835.  7.35e2 1020.  6.18e+2 1.26e3
5 steps                    627 6899.  3695.  6513.  4.41e3 9069.  1.49e+3 1.29e4
6 very_active_minutes      627   22.1   21.8   15.7 6.48e0   29.5 6.50e-1 6.62e1


In [ ]:
miss_gap_func(df = pca_activity, path.out =  '/home/jupyter/workspaces/result/Prostate/')

In [8]:
path.out = '/home/jupyter/workspaces/Data/'
cohort = 'Prostate'
df = pca_activity_sex
key_metrics <- c("steps", "calories_out", "sedentary_minutes",
                    "lightly_active_minutes", "fairly_active_minutes",
                    "very_active_minutes", "sex_at_birth")

    patient_stats <- df |>
    group_by(person_id) |>
    summarise(
        n_obs = n(),
        across(all_of(key_metrics), list(
        mean = ~mean(.x, na.rm = TRUE),
        sd   = ~sd(.x,   na.rm = TRUE),
        cv   = ~ifelse(mean(.x, na.rm = TRUE) > 0,
                        sd(.x, na.rm = TRUE) / mean(.x, na.rm = TRUE) * 100, NA)
        ), .names = "{.col}__{.fn}"),
        .groups = "drop"
    )
    patient_stats$cohort <- cohort
    write.table(patient_stats, paste0(path.out, cohort,'_activity_summary_stats.txt'), sep = '\t', quote = F, row.names = F)

    # ── 2. Summary table: distribution of patient means ────────────────────────
    mean_cols <- paste0(key_metrics, "__mean")

    cohort_summary <- patient_stats |>
    select(all_of(mean_cols)) |>
    pivot_longer(everything(), names_to = "metric", values_to = "value") |>
    mutate(metric = str_remove(metric, "__mean")) |>
    group_by(metric) |>
    summarise(
        N        = sum(!is.na(value)),
        Mean     = mean(value,              na.rm = TRUE),
        SD       = sd(value,                na.rm = TRUE),
        Median   = median(value,            na.rm = TRUE),
        Q1       = quantile(value, 0.25,    na.rm = TRUE),
        Q3       = quantile(value, 0.75,    na.rm = TRUE),
        P5       = quantile(value, 0.05,    na.rm = TRUE),
        P95      = quantile(value, 0.95,    na.rm = TRUE),
        .groups  = "drop"
    )
    print(cohort_summary, n = Inf)

Warning message:
“There were 1881 warnings in `summarise()`.
The first warning was:
ℹ In argument: `across(...)`.
ℹ In group 1: `person_id = 1002338`.
Caused by warning in `mean.default()`:
! argument is not numeric or logical: returning NA
ℹ Run `dplyr::last_dplyr_warnings()` to see the 1880 remaining warnings.”


# A tibble: 7 × 9
  metric                  N   Mean     SD Median      Q1     Q3       P5     P95
  <chr>               <int>  <dbl>  <dbl>  <dbl>   <dbl>  <dbl>    <dbl>   <dbl>
1 calories_out          627 2529.   438.  2507.  2239.   2797.  1855.     3263. 
2 fairly_active_minu…   627   18.6   15.1   15.6    7.49   26.5    1.50     47.9
3 lightly_active_min…   627  173.    67.0  173.   128.    218.    65.9     283. 
4 sedentary_minutes     627  882.   197.   835.   735.   1020.   618.     1255. 
5 sex_at_birth            0  NaN     NA     NA     NA      NA     NA        NA  
6 steps                 627 6899.  3695.  6513.  4411.   9069.  1495.    12929. 
7 very_active_minutes   627   22.1   21.8   15.7    6.48   29.5    0.650    66.2


In [9]:
table(pca_activity_sex$sex_at_birth)


  Male 
449924 